<a href="https://colab.research.google.com/github/AntonRize/WILL/blob/main/%5CColab_Notebooks%5CROM_M_sin_i.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

R.O.M. HOLOGRAPHIC DECODER  -  breaking the M sin(i) degeneracy in closed form
==============================================================================

Strong claim under test: from the RAW spectroscopic shift Z_raw(o) alone (plus
the orbit shape e), the R.O.M. closed algebra recovers the true kinematic
projection beta, the argument of periapsis omega_i, AND the inclination i --
WITHOUT knowing i, M, or G in advance, and without differential equations.

We test it the only honest way: FORWARD-SYNTHESISE the observable from a known
truth, then run a BLIND INVERSE that is not allowed to see i, beta, M or G, and
check it returns the truth.

Model (ROM, as given):
  beta_los(o) = (beta/sqrt(1-e^2)) (cos(o+wi) + e cos wi) sin i   # line-of-sight Doppler
  kappa_o^2(o)= 2 beta^2 (1 + e cos o)/(1-e^2)                     # local potential (energy invariant)
  beta_o^2(o) =   beta^2 (1+e^2+2e cos o)/(1-e^2)                  # local kinetic
  tau(o)      = sqrt( (1-kappa_o^2)(1-beta_o^2) )                  # systemic spacetime factor
  Z_sys(o)    = 1/tau(o);   Z_raw(o) = (1 + beta_los(o)) * Z_sys(o)

True S0-2 (GRAVITY): e=0.88466, wi=66.13 deg, beta=0.00645, i=134.0 deg.

In [1]:
import mpmath as mp
mp.mp.dps = 40
pi = mp.pi
sin, cos, sqrt, acos, asin = mp.sin, mp.cos, mp.sqrt, mp.acos, mp.asin

G = mp.mpf('6.67430e-11'); c = mp.mpf('2.99792458e8'); Msun = mp.mpf('1.98892e30')

# ---------------- TRUTH (hidden from the inverse) ----------------
e_true  = mp.mpf('0.88466')
wi_true = mp.mpf('66.13')*pi/180
b_true  = mp.mpf('0.00645')
i_true  = mp.mpf('134.0')*pi/180
T_S2    = mp.mpf('16.05')*mp.mpf('3.1557e7')   # 16.05 yr -> s (final scale only)

def make_model(beta, e, wi, inc):
    bint = beta/sqrt(1-e**2)
    def beta_los(o): return bint*(cos(o+wi)+e*cos(wi))*sin(inc)
    def kappa_o2(o): return 2*beta**2*(1+e*cos(o))/(1-e**2)
    def beta_o2(o):  return beta**2*(1+e**2+2*e*cos(o))/(1-e**2)
    def tau(o):      return sqrt((1-kappa_o2(o))*(1-beta_o2(o)))
    def Zraw(o):     return (1+beta_los(o))/tau(o)
    return beta_los, kappa_o2, beta_o2, tau, Zraw

beta_los, kappa_o2, beta_o2, tau, Zraw = make_model(b_true, e_true, wi_true, i_true)
def fmt(x, n=6): return mp.nstr(x, n)
PASS = []

print("="*78); print("STEP 1  FORWARD SYNTHESIS  (reproduce the paper's S0-2 numbers)"); print("="*78)
Oe1 = -wi_true; Oe2 = pi - wi_true
A_raw, B_raw = Zraw(Oe1), Zraw(Oe2)
tau_e1, tau_e2 = tau(Oe1), tau(Oe2)
print("  tau(-wi)      = %s   (paper 0.999501)" % fmt(tau_e1))
print("  tau(pi-wi)    = %s   (paper 0.999775)" % fmt(tau_e2))
print("  Z_raw(-wi)    = %s   (paper 1.014019)" % fmt(A_raw))
print("  Z_raw(pi-wi)  = %s   (paper 0.993834)" % fmt(B_raw))
chk = (abs(tau_e1-mp.mpf('0.999501'))<5e-6 and abs(A_raw-mp.mpf('1.014019'))<5e-6
       and abs(B_raw-mp.mpf('0.993834'))<5e-6)
PASS.append(("reproduces paper extrema", chk)); print("  -> matches paper:", chk)

print("\n"+"="*78); print("STEP 2  DECRYPTION INVARIANT = 2  (forward identity)"); print("="*78)
c_w = cos(wi_true)
inv = A_raw*tau_e1*(1-e_true*c_w) + B_raw*tau_e2*(1+e_true*c_w)
print("  Z_raw(-wi)tau(-wi)(1-e cos wi) + Z_raw(pi-wi)tau(pi-wi)(1+e cos wi) = %s" % mp.nstr(inv,20))
PASS.append(("Decryption Invariant = 2", abs(inv-2) < mp.mpf('1e-30'))); print("  -> = 2 exactly:", abs(inv-2) < mp.mpf('1e-30'))

print("\n"+"="*78); print("STEP 3  BALANCE NODES (beta_los=0) + verify factorizations"); print("="*78)
u0 = acos(-e_true*cos(wi_true))
ob_plus  = (u0 - wi_true)
ob_minus = (-u0 - wi_true)
Theta = sqrt(1 - e_true**2*cos(wi_true)**2)
for name, ob, sign in [("node+", ob_plus, +1), ("node-", ob_minus, -1)]:
    los = beta_los(ob)
    fac = 1 + e_true*cos(ob)
    fac_pred = Theta*(Theta + sign*e_true*sin(wi_true))
    k2 = kappa_o2(ob); k2p = 2*b_true**2/(1-e_true**2)*Theta*(Theta+sign*e_true*sin(wi_true))
    b2 = beta_o2(ob);  b2p = b_true**2/(1-e_true**2)*(Theta+sign*e_true*sin(wi_true))**2
    ok = (abs(los)<1e-30 and abs(fac-fac_pred)<1e-30 and abs(k2-k2p)<1e-30 and abs(b2-b2p)<1e-30)
    PASS.append((name+" factorization", ok))
    print("  %s o=%8.3f deg : beta_los=%.1e  factorization ok=%s  perfect-square ok=%s"
          % (name, float(ob*180/pi), float(los), abs(fac-fac_pred)<1e-30, abs(b2-b2p)<1e-30))
Nplus, Nminus = Zraw(ob_plus), Zraw(ob_minus)
print("  N+ = Z_raw(node+) = %s   N- = Z_raw(node-) = %s" % (mp.nstr(Nplus,12), mp.nstr(Nminus,12)))
print("  node signal |N+ - N-| = %.2e   <- precision a real measurement must beat" % float(abs(Nplus-Nminus)))

print("\n"+"="*78); print("STEP 4  BLIND INVERSE (no i, no beta, no M, no G given)"); print("="*78)
print("  Known to solver: e, Z_raw at 2 nodes (N+,N-) and 2 extrema (A,B).")
def node_inv2(b, w, sign, e):
    Th = sqrt(1 - e**2*cos(w)**2); g = Th + sign*e*sin(w)
    return (1 - 2*b**2*Th*g/(1-e**2))*(1 - b**2*g**2/(1-e**2))
def resid(b, w):
    return [node_inv2(b, w, +1, e_true) - 1/Nplus**2,
            node_inv2(b, w, -1, e_true) - 1/Nminus**2]
b0, w0 = mp.mpf('0.004'), mp.mpf('50')*pi/180
sol = mp.findroot(lambda b, w: resid(b, w), (b0, w0))
b_rec, w_rec = sol[0], sol[1]
print("  start guess : beta=%.5f  wi=%.2f deg" % (float(b0), float(w0*180/pi)))
print("  RECOVERED   : beta=%.10f  wi=%.6f deg" % (float(b_rec), float(w_rec*180/pi)))
print("  TRUTH       : beta=%.10f  wi=%.6f deg" % (float(b_true), float(wi_true*180/pi)))
ok_b = abs(b_rec-b_true)<mp.mpf('1e-12'); ok_w = abs(w_rec-wi_true)<mp.mpf('1e-12')
PASS.append(("blind beta recovery", ok_b)); PASS.append(("blind omega_i recovery", ok_w))
print("  -> beta recovered:", ok_b, "  omega_i recovered:", ok_w)

def tau_from(b, w, o, e):
    k2 = 2*b**2*(1+e*cos(o))/(1-e**2); b2 = b**2*(1+e**2+2*e*cos(o))/(1-e**2)
    return sqrt((1-k2)*(1-b2))
te1 = tau_from(b_rec, w_rec, -w_rec, e_true)
te2 = tau_from(b_rec, w_rec, pi-w_rec, e_true)
sini_rec = sqrt(1-e_true**2)*(A_raw*te1 - B_raw*te2)/(2*b_rec)
print("  sin i extracted = %s   (true %s)" % (mp.nstr(sini_rec,10), mp.nstr(sin(i_true),10)))
ok_sini = abs(sini_rec-sin(i_true))<mp.mpf('1e-10'); PASS.append(("blind sin i recovery", ok_sini))
i_a = asin(sini_rec)*180/pi
print("  -> sin i recovered:", ok_sini, " => i = %.3f or %.3f deg (quadrant from astrometry)" % (float(i_a), 180-float(i_a)))

print("\n"+"="*78); print("STEP 5  CLOSE SYSTEM (a, R_s, M as OUTPUTS; G only -> kg)"); print("="*78)
a  = T_S2*b_rec*c/(2*pi); Rs = 2*b_rec**2*a; GM = b_rec**2*c**2*a; M = Rs*c**2/(2*G)
print("  a   = T beta c/2pi  = %.4e m = %.1f AU  (S0-2 ~ 1020 AU)" % (float(a), float(a/mp.mpf('1.495978707e11'))))
print("  R_s = 2 beta^2 a    = %.4e m" % float(Rs))
print("  GM  = beta^2 c^2 a  = %.4e m^3/s^2  (no G used)" % float(GM))
print("  M   = R_s c^2/(2G)  = %.4e kg = %.3e Msun  (Sgr A* ~ 4.3e6 Msun)" % (float(M), float(M/Msun)))

print("\n"+"="*78); print("STEP 6  ALTERNATIVE (astrometry gives e,wi): 1 node + 1 extremum"); print("="*78)
b_node = mp.findroot(lambda b: node_inv2(b, wi_true, +1, e_true) - 1/Nplus**2, mp.mpf('0.004'))
sini_b = sqrt(1-e_true**2)*(A_raw*tau_from(b_node, wi_true, -wi_true, e_true)
                            - B_raw*tau_from(b_node, wi_true, pi-wi_true, e_true))/(2*b_node)
print("  beta (1 node, wi known) = %.10f (truth %.10f) ok=%s"
      % (float(b_node), float(b_true), abs(b_node-b_true)<mp.mpf('1e-12')))
print("  sin i (1 extremum)      = %.10f (truth %.10f) ok=%s"
      % (float(sini_b), float(sin(i_true)), abs(sini_b-sin(i_true))<mp.mpf('1e-10')))
PASS.append(("astrometry-assisted 2-point method",
             abs(b_node-b_true)<mp.mpf('1e-12') and abs(sini_b-sin(i_true))<mp.mpf('1e-10')))

print("\n"+"="*78); print("SUMMARY"); print("="*78)
allok = True
for name, ok in PASS:
    allok &= ok; print("  [%s] %s" % ("PASS" if ok else "FAIL", name))
print("\n  Inputs used by inverse : e, Z_raw at 2 nodes + 2 extrema, (T for final scale)")
print("  NOT used               : i, beta, M, G, any differential equation")
print("  i, M, G are OUTPUTS.   OVERALL: %s" % ("ALL PASS" if allok else "SOME FAILED"))
print("\n  Honest caveats:")
print("   * sin i fixes i up to i <-> 180-i; astrometry picks the quadrant.")
print("   * the node signal carrying beta,wi is O(beta^2); here |N+ - N-| = %.2e," % float(abs(Nplus-Nminus)))
print("     so real data must measure Z_sys to ~that level -> the SNR/e/coverage thresholds.")
print("   * noiseless synthesis: proves the algebra inverts and is i/M/G-free, not noise robustness.")

STEP 1  FORWARD SYNTHESIS  (reproduce the paper's S0-2 numbers)
  tau(-wi)      = 0.999501   (paper 0.999501)
  tau(pi-wi)    = 0.999775   (paper 0.999775)
  Z_raw(-wi)    = 1.01402   (paper 1.014019)
  Z_raw(pi-wi)  = 0.993835   (paper 0.993834)
  -> matches paper: True

STEP 2  DECRYPTION INVARIANT = 2  (forward identity)
  Z_raw(-wi)tau(-wi)(1-e cos wi) + Z_raw(pi-wi)tau(pi-wi)(1+e cos wi) = 2.0
  -> = 2 exactly: True

STEP 3  BALANCE NODES (beta_los=0) + verify factorizations
  node+ o=  44.847 deg : beta_los=5.7e-44  factorization ok=True  perfect-square ok=True
  node- o=-177.107 deg : beta_los=5.7e-44  factorization ok=True  perfect-square ok=True
  N+ = Z_raw(node+) = 1.00060241062   N- = Z_raw(node-) = 1.00002377972
  node signal |N+ - N-| = 5.79e-04   <- precision a real measurement must beat

STEP 4  BLIND INVERSE (no i, no beta, no M, no G given)
  Known to solver: e, Z_raw at 2 nodes (N+,N-) and 2 extrema (A,B).
  start guess : beta=0.00400  wi=50.00 deg
  RECOVERED   : be